Sebastian Gonzales-Portillo\
University of Miami

## 1. Introduction

Through the Kepler and TESS missions, NASA has been collecting immense amounts of data about stars in the galaxy. Some of this data includes the flux (i.e luminosity) of stars over time. By carefully analyzing this data, we can detect dips in flux that are ideally periodic which can indicate an exoplanet that is in orbit (since a celestial object crossing the visibile area of the light source would effectively "dim" its intensity).

## 2. Methods

We will be implementing some methods to see if we can indentify a transitory period in our time-series data. It is important to understand that these transitory signals we are looking for are "box-shaped" and are usually short/small. From this, we know which methods will be better posed to solve this problem but nonetheless we will explore how these different methods interact with the data. Some of the methods we will look at include:

1. **Fourier Transform**
2. **Regularlization** (more on that later)
3. **SVD**
4. **BLS**

The Fourier Transform decomposes a time-series signal into sinusoidal components of varying frequencies. It could potentially be useful in identifying periodic signals. 
\
Regularization techniques, such as Tikhonov (L2) or L1 regularization, help reduce noise and improve the reconstruction of signals by penalizing large variations in the residuals. This can aid in smoothing the data and isolating transitory features for further analysis.
\
SVD decomposes the time-series data into orthogonal components ranked by their significance. By retaining only a subset of the most significant components, we can denoise the data and highlight periodic patterns. However, SVD's effectiveness depends on the nature of the data and the number of components retained.
\
The BLS method is specifically designed for identifying periodic box-like signals, such as exoplanet transits. It models the signal as a sequence of box-shaped dips in flux and computes the period that best fits these features, making it highly effective for detecting transitory events in the data. This will likely be where we will strike gold.

## 3. Analysis

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt


### Retrive data

In [ ]:
from lksearch import TESSSearch, KeplerSearch, MASTSearch
import lightkurve as lk
from lightkurve import search_targetpixelfile

search = TESSSearch('TOI 1161')
search # data for this star

In [ ]:
tpf = search_targetpixelfile('TIC 158324245', mission='TESS', sector=14).download()


In [ ]:
%matplotlib inline
tpf.plot()

This is the particular pixel frame for this star at this moment at time

In [ ]:
tpf.time, tpf.flux 


In [ ]:
tpf.time.shape, tpf.flux.shape

In [ ]:
lc = tpf.to_lightcurve(aperture_mask=tpf.pipeline_mask);
lc.plot()

In [ ]:
lc_normalized = lc.normalize()
lc_normalized.plot()

In [ ]:
lc_norm_flat = lc_normalized.flatten()
lc_norm_flat.plot()

Use the built in periodagram to identify the dominant period

In [ ]:
periodagram = lc_norm_flat.to_periodogram(method="bls", period=np.arange(1, 10, 1e-2))
periodagram.plot()
best_max_period = periodagram.period_at_max_power
print('Best fit period: {:.3f}'.format(best_max_period))

In [ ]:
folded_lc = lc_norm_flat.fold(period=best_max_period)
folded_lc.scatter(s=0.1)

The above does seem promising, but it is still quite noisy and the selected period is not close enough to show the separation we want. Let us try different tehniques to see if we can get a refined (or in general a better) dominant period.

Lets first use the Fourier transform to identify a dominant period. It will decompose the time-series data into sinusoidal components. This will likely require that the transit signals be strong enough, let us see if it can get the job done. We need to ensure we have removed long term trends by using a filtet such as the Savitzky-Golay filter, which smoothens the data like a regualr moving average but maintains the shape and height of peaks by fitting a polynomial to the window size by using least squares method from which the centroid of the polynomial is taken as our smoothed value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.fft import fft, fftfreq

time = lc_norm_flat.time.value
flux = lc_norm_flat.flux.value


mask = ~np.isnan(time) & ~np.isnan(flux)
time = time[mask]
flux = flux[mask]

#Detrend the flux using Savitzky-Golay filter
flux_trend = savgol_filter(flux, window_length=101, polyorder=3)  # Adjust window_length based on data
flux_detrended = flux - flux_trend  # Remove the long-term trend

# apply Fourier Transform
N = len(time)
dt = np.median(np.diff(time))  # Sampling interval (median of time difaferences)
freq = fftfreq(N, dt)          # Frequency axis
power = np.abs(fft(flux))**2  # Power spectrum

# Only keep positive frequencies
positive_freq = freq > 0
freq = freq[positive_freq]
power = power[positive_freq]

#Identify the dominant period
best_frequency = freq[np.argmax(power)]
best_period = 1 / best_frequency
print(f"Best Frequency: {best_frequency:.4f} [1/days]")
print(f"Best Period: {best_period:.4f} days")

#Plot the Fourier Transform result
plt.figure(figsize=(10, 5))
plt.plot(freq, power, label="Power Spectrum")
plt.axvline(best_frequency, color='red', linestyle='--', label=f'Best Period = {best_period:.4f} days')
plt.xlabel("Frequency [1/days]")
plt.ylabel("Power")
plt.title("Fourier Transform of Light Curve")
plt.legend()
plt.show()


# Convert time into phase using the best period
phase = (time % best_period) / best_period
phase = np.concatenate([phase, phase + 1])  # To wrap around the phase plot
flux_folded = np.concatenate([flux, flux])


# Plot the folded light curve
plt.figure(figsize=(10, 5))
plt.scatter(phase, flux_folded, s=2, alpha=0.7, label="Folded Light Curve")
plt.xlabel("Phase (Folded)")
plt.ylabel("Normalized Flux")
plt.title("Folded Light Curve Using Best Period")
plt.legend()
plt.show()

lc.fold(period=best_period).scatter()


The found period indicates something that is likely not an exoplanet. Clearly, the Fourier transform failed to find a signal that works for the data. As mentioned, exoplanet transitions typically produce box-shaped signals, which is effectively a square wave. The Fourier transform thus takes the square wave and decomposes it into a series of sinusoidal harmonics. This is because the flux (relatively) remains constant, but when it has an object in front of it the flux rapidly takes a decrease to a new (relatively) constant level. Thus we have a periodic signal that alternates between two levels with very sharp transitions that then require very high frequency harmonics to reconstruct these edges leading to power being distrbuted. Transit signals are usually very weak compared to other noise picked up by the instrument. Thus the fundamental frequency (i.e. the period of the transit) is diluted across harmonics and the sharp edges introduce high-frequency noise due to the power being split by the the f.f. and its higher harmonics. Moreover, given that the data likely contains numerous stochaistic noise from the star, the instrument itself, or the surroundings, we would need further advanced noise reduction techniques to properly the signal.

Next, we will use Singular Value Decomposition (SVD) to examine the singular values of the data

In [ ]:
from scipy.signal import savgol_filter
from scipy.linalg import svd
from lightkurve import LightCurve

time = lc_norm_flat.time.value
flux = lc_norm_flat.flux.value

# Mask NaN values
mask = ~np.isnan(time) & ~np.isnan(flux)
time = time[mask]
flux = flux[mask]


# Reshape the flux into overlapping windows for SVD
window_size = 100  # Adjust as needed
reshaped_flux = np.array([flux[i:i + window_size] for i in range(len(flux) - window_size)])

# Perform SVD
U, s, Vt = svd(reshaped_flux, full_matrices=False)

# Test multiple values of k
for k in [2,4,5,7,8,9,10,15,20]:  # Adjust k to experiment with retained components
    # Reconstruct the flux using the top k singular values
    reconstructed_flux_matrix = np.dot(U[:, :k], np.dot(np.diag(s[:k]), Vt[:k, :]))

    # Combine the overlapping windows
    reconstructed_flux = np.zeros(len(flux))
    counts = np.zeros(len(flux))  # Count contributions at each time point

    for i in range(reconstructed_flux_matrix.shape[0]):
        reconstructed_flux[i:i + window_size] += reconstructed_flux_matrix[i]
        counts[i:i + window_size] += 1

    # Normalize by the number of overlaps
    reconstructed_flux /= counts

    # Plot the original and reconstructed flux
    plt.figure(figsize=(10, 5))
    plt.plot(time, flux, label="Original Flux (Normalized)", alpha=0.5)
    plt.plot(time, reconstructed_flux, label=f"Reconstructed Flux (k={k})", color="orange")
    plt.xlabel("Time [days]")
    plt.ylabel("Flux")
    plt.title(f"SVD Denoising and Reconstruction (k={k})")
    plt.legend()
    plt.show()

    # Phase-fold the reconstructed signal using the best period from SVD
    best_period = 1 / best_frequency  # Assuming best_frequency is calculated elsewhere
    phase = (time % best_period) / best_period

    # Create LightCurve object for folding using Lightkurve
    lightcurve = LightCurve(time=time, flux=reconstructed_flux)
    folded_lc = lightcurve.fold(period=best_period)
    # Plot using Lightkurve folding
    folded_lc.scatter()
    plt.title(f"Folded Light Curve Using Lightkurve (k={k})")
    plt.xlabel("Phase")
    plt.ylabel("Flux")
    plt.show()

    # Plot the phase-folded light curve
    phase = np.concatenate([phase, phase + 1])  # Wrap around the phase
    flux_folded = np.concatenate([reconstructed_flux, reconstructed_flux])

    plt.figure(figsize=(10, 5))
    plt.scatter(phase, flux_folded, s=2, alpha=0.7, label=f"Folded Light Curve (SVD, k={k})")
    plt.xlabel("Phase (Folded)")
    plt.ylabel("Normalized Flux")
    plt.title(f"Folded Light Curve Using SVD Reconstruction (k={k})")
    plt.legend()
    plt.show()


SVD captures the largest variability in the data, which within this context is likely stellar variability. In other words keep a small $k$ singular values will capture large-scale variability leading to a smoother reconstucted flux but emphasizes larger-scale trends which will miss the transit trends we look for. If we pick a bigger $k$, we are able to retain more features and finer details but might capture more noise.

Next, we will use a combination of techniques to attempt to achieve a better result. First, we will apply a Savitzky-Golay filter to smoothen the data and remove long-term trends by fitting successive polynomial functions, which is similar to a moving average but it better retains the shape and edges of the original flux. \
Then we will use SVD, and based off the previous computations pick $k=6$ since it is a good middle ground that will help remove noise and preserve dominant patterns
$$
A =  U \Sigma V^T \\
\downarrow \\
A_k =   U[:,:k]\Sigma[:k,:k]V^T[:k,:]
$$
$A_k$ will be our (roughly) denoised signal.\
I thought of using Tikhonov regularization that will minimize the residuals between the reconstructed flux and the original flux but it should penalize abrupt changes in the resiudal ($\alpha \Sigma(\nabla r)^2$) to enforce smoothness which will smooth out the box shaped dips we are looking for. Instead we can try use an L1 regularization ($\alpha \Sigma |\nabla r|$)that will instead promote sparsity in the changes while reducing noise.

\
Finally,  the Fourier trasnform is applied to the residual flux to identify periodicities.
$$
\text{Power} = | \mathcal{F} ( \text{flux residual} )|^2

\\
\text{Period} = \frac{1}{\text{Frequency}}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd
from scipy.signal import savgol_filter
from scipy.optimize import minimize
from scipy.fft import fft, fftfreq

# Functions
def l1_regularization(reconstructed_flux, original_flux, alpha=0.1):
    """Apply L1 regularization to refine the residuals."""
    def cost_function(residual):
        return np.sum((original_flux - (reconstructed_flux + residual))**2) + alpha * np.sum(np.abs(np.diff(residual)))
    initial_guess = np.zeros_like(reconstructed_flux)
    result = minimize(cost_function, initial_guess)
    return result.x

# Data preprocessing
time = lc_norm_flat.time.value
flux = lc_norm_flat.flux.value

# Remove NaN values
mask = ~np.isnan(time) & ~np.isnan(flux)
time = time[mask]
flux = flux[mask]


# Define SVD parameters
window_size = 50

# Reshape flux for SVD (use overlapping windows)
reshaped_flux = np.array([flux[i:i+window_size] for i in range(len(flux) - window_size)])

# Perform SVD
U, s, Vt = svd(reshaped_flux, full_matrices=False)

# Reconstruct the flux signal using the top k singular values
k = 6 # Number of singular values to retain
reconstructed_flux_matrix = np.dot(U[:, :k], np.dot(np.diag(s[:k]), Vt[:k, :]))

# Combine the overlapping windows
reconstructed_flux = np.zeros(len(flux))
counts = np.zeros(len(flux))
for i in range(len(reconstructed_flux_matrix)):
    reconstructed_flux[i:i+window_size] += reconstructed_flux_matrix[i]
    counts[i:i+window_size] += 1
reconstructed_flux /= counts

# Apply Tikhonov Regularization
residual = l1_regularization(reconstructed_flux, flux, alpha=0.1)
reconstructed_flux += residual

# Detrend the reconstructed flux using Savitzky-Golay filter
reconstructed_flux_detrended = savgol_filter(reconstructed_flux, window_length=101, polyorder=3)

# Fourier Transform to identify best period
residual_flux = flux - reconstructed_flux
N = len(residual_flux)
dt = np.median(np.diff(time))  # Sampling interval
freq = fftfreq(N, dt)
power = np.abs(fft(residual_flux))**2
positive_freq = freq > 0
freq = freq[positive_freq]
power = power[positive_freq]

# Identify the dominant frequency and period
best_frequency = freq[np.argmax(power)]
best_period = 1 / best_frequency
print(f"Best Frequency (Fourier): {best_frequency:.4f} [1/days]")
print(f"Best Period (Fourier): {best_period:.4f} days")

# Plot the Fourier Transform result
plt.figure(figsize=(10, 5))
plt.plot(freq, power, label="Power Spectrum")
plt.axvline(best_frequency, color='red', linestyle='--', label=f'Best Period = {best_period:.4f} days')
plt.xlabel("Frequency [1/days]")
plt.ylabel("Power")
plt.title("Fourier Transform of Residuals")
plt.legend()
plt.show()

# Phase fold the light curve with the best period
phase = (time % best_period) / best_period
phase = np.concatenate([phase, phase + 1])  # Wrap around phase plot
flux_folded = np.concatenate([reconstructed_flux_detrended, reconstructed_flux_detrended])

# Create LightCurve object for folding using Lightkurve
lightcurve = LightCurve(time=time, flux=reconstructed_flux)
folded_lc = lightcurve.fold(period=best_period)
# Plot using Lightkurve folding
folded_lc.scatter()
plt.title(f"Folded Light Curve Using Lightkurve (k={k})")
plt.xlabel("Phase")
plt.ylabel("Flux (normalized)")
plt.show()


# Plot the folded light curve
plt.figure(figsize=(10, 5))
plt.scatter(phase, flux_folded, s=2, alpha=0.7, label="Folded Light Curve (SVD)")
plt.xlabel("Phase (Folded)")
plt.ylabel("Normalized Flux")
plt.title("Folded Light Curve Using SVD Reconstruction")
plt.legend()
plt.show()

Although we are indeed capturing a strong periodic transitory signal, according to the ExoFOP website there is no known celestial object with the given period for this star. While it is possible that we have detected a planet with an approximate 26 day periods, we would need more evidence to deduce this. So it is currently unclear to deduce we have detected a period correlated to an exoplanet. It is possible this change in flux could be due to the stellar rotation in which we are seeing stellar spots or magnetic activity. To address this, we could include higher values of $k$ to ensure we don't lose out on the subtle features of a transit period, but it would require advanced noise filtering techniques to account for the added noise. We know  that SVD limits our approach since it captures smooth, low-rank approximations of the flux that can smoothen sharper edges of the box-shaped transit signals. The same problem mentioned before with the Fourier transform still applies. Overall, we would need more advanced detrending, noise filtering, and fine-tuning of our parameters to ensure we capture the desired signal (we know that there is a planet with a period of approx. 1.7 days which I will show next.)

Since we know we are looking for square-shaped waves, we can use an approach that is better suited to this type of signal. The **Box-Least-Squares** (BLS) method is designed to detect box-shaped periodic signals, which are more characteristic of exoplanetary transits, compared to sinusoidal signals detected by methods like the Fourier transform.


The BLS method models the light curve using a periodic box function and minimizes the least-square residuals between the observed light curve and the box model. For a given trial period $P_k$, we have:

$$
   \chi^2(P_k) = \sum_{i=1}^N \left(F_i - F_{\text{model}, i}\right)^2,
$$

Then for each $P_k$ we can compute the BLS power:
$$
\text{Power}(P_k) = \frac{\Delta^2 T}{\sigma^2},
$$
where:
- $\Delta$: Transit depth,
- $T$: Transit duration,
- $\sigma^2$: Variance of residuals outside the transit.

By scanning over a range of trial periods, the BLS algorithm identifies the period that best minimizes this  (i.e. max power), thus isolating periodic box-shaped features in the data. This method is particularly well-suited for capturing the sharp drops and plateaus in flux that occur during an exoplanet transit, which is a departure from the smooth, sinusoidal features targeted by Fourier methods.




In [ ]:
# Use Box Least Squares (BLS) to refine the period
from astropy.timeseries import BoxLeastSquares

bls = BoxLeastSquares(lc_norm_flat.time.value, lc_norm_flat.flux.value)
results = bls.autopower(0.1)  # Specify the duration fraction range for the transit

# Find the refined period
refined_period = results.period[np.argmax(results.power)]
print(f"Refined Period: {refined_period:.4f} days")

# Plot the BLS power spectrum
plt.figure(figsize=(10, 5))
plt.plot(results.period, results.power, label="BLS Power Spectrum")
plt.axvline(refined_period, color='r', linestyle='--', label=f"Best Period = {refined_period:.4f} days")
plt.xlabel("Period [days]")
plt.ylabel("Power")
plt.title("BLS Period Detection")
plt.legend()
plt.show()

In [ ]:
folded_lc = lc_norm_flat.fold(period=refined_period)
folded_lc.scatter(s=.5)

This looks much better, and we can see that there is a clear separation from the rest of the data

In [ ]:
binned_lc = folded_lc.bin(binsize=10)
binned_lc.scatter(s=1)

In [ ]:
plt.figure(figsize=(10, 5))
binned_lc.scatter(s=5, color='purple')
plt.xlim(0.2, 0.3)  # Zoom into the transit region
plt.ylim(0.9975, 1.001)  # Narrow the flux range
plt.xlabel("Phase (Folded)")
plt.ylabel("Normalized Flux")
plt.title("Zoomed View of Transit")
plt.show()


In [ ]:
depth = 1 - np.min(binned_lc.flux)
print(f"Transit Depth: {depth * 100:.2f}%")

This data is highly suggestive a planet in transit! Likely a smaller one too (or perhaps just extremely distant large exoplanet). We see the flux dipping and then recovering in quite smooth way.
It is very probable that this planet/celestial body has the comptued transit period of $\approx 1.7636$ days. Now, it is important to remember we would need even further steps to fully validate this is indeed an exoplanet. But after looking up in the ExoFOP database, we can see that for star TIC 158324245 there is indeed a confirmed exoplanet with period 1.763588 named KOI-13 b!

## 4. Conclusion
We explored various methods to identify transitory periods in time-series data from NASA's TESS mission, particularly focusing on detecting exoplanetary transits characterized by square-shaped flux drops.

The Fourier Transform, while effective for detecting sinusoidal signals, proved inadequate for capturing the box-like shapes of transits. SVD offered a valuable denoising and dimensionality reduction framework, allowing us to reconstruct the signal and highlight periodic features. However, it still struggled with the sharp transitions inherent to transit signals.

The Box Least Squares (BLS) method emerged as the most suitable approach for this problem. By tailoring the model to the box-like nature of transit signals, BLS effectively isolated periods corresponding to potential exoplanet transits, providing a significant improvement over other methods. Its ability to minimize residuals between the observed data and the box model ensured a strong alignment with the expected signal structure.

Through this project, I explored how different methods interacted with the specific conditions of our problem. In the future, I aim to investigate more combinations and refinements of these methods that might yield inteesting results. Additionally, I hope to explore the application of machine learning models to automate the detection of these signals, as I think this could be a very cool project and deepen my understanding of the complex math regarding this problem.